# Curadoria do dataset TEIA

**16 cortes de teste + 1 referência derivada incluída no dataset, todos com 10 segundos.**

No VS Code, escolha **Select Kernel → Python Environments → `soundSamples/.venv/bin/python`**.
Execute as seções 1–3; na seção 4 revise **uma fonte por vez**, escute, ajuste os inícios se
precisar e marque a aprovação. Só a seção 5 exporta os WAVs definitivos.

“Mais ativo” significa maior proporção de sinal acima de um limiar, não melhor música,
monofonia ou isolamento acústico. A seleção favorece atividade e pode deixar passagens
suaves de fora; a escuta tem a decisão final. Não há análise perceptiva de loudness aqui.

Os originais nunca são modificados ou apagados. Leia `PROVENIENCIA.md` para créditos e
condições de uso. O arquivo curto `Original.wav` não é uma décima fonte independente:
a referência exportada vem dos primeiros 10 segundos de `Excerpt 1.aif`.

**Seleção atual:** `duos_trumpet_guitar` foi excluído a pedido do usuário. Os primeiros 10 segundos de Excerpt 1 entram como `Original.wav`, além dos 16 cortes; a referência pode alimentar os testes, embora derive de uma fonte já representada.


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import platform
import os
import tempfile
import re
from datetime import datetime, timezone

import numpy as np
import soundfile as sf
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "teia-matplotlib-cache"))
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
from IPython.display import Audio, Markdown, display, clear_output
from IPython import get_ipython

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

# Funciona abrindo o notebook da raiz do projeto ou da própria pasta soundSamples.
ROOT = Path.cwd().resolve()
if ROOT.name != "soundSamples":
    ROOT = ROOT / "soundSamples"
if not ROOT.is_dir():
    raise FileNotFoundError("Abra o notebook a partir do projeto TEIA ou de soundSamples.")

VERSION = "v1"  # Para nova seleção/exportação, use v2, v3, etc.
if not re.fullmatch(r"[A-Za-z0-9_-]+", VERSION):
    raise ValueError("VERSION deve conter apenas letras, números, _ ou -.")
DATASET = ROOT / "dataset"
DRAFT = DATASET / f"revisao_{VERSION}.json"
OUTPUT = DATASET / VERSION
SELECTION = dict(duration_s=10.0, step_s=0.5, block_s=0.1,
                 threshold_floor_dbfs=-70.0, threshold_below_p95_db=40.0,
                 algorithm="active_fraction_then_rms_then_earliest_v1")
PREPARATION = dict(sample_rate=48000, channels=1, subtype="PCM_24", fade_ms=5.0,
                   target_rms_dbfs=-24.0, peak_ceiling_dbfs=-1.0,
                   downmix="arithmetic_mean", resample="scipy.signal.resample_poly",
                   resample_window=["kaiser", 5.0], resample_padtype="constant")
print("Ambiente:", platform.python_version(), "|", ROOT)
print("Revisão:", DRAFT.name, "| Exportação:", OUTPUT)

In [ ]:
# As tomadas de Edimburgo foram transcritas das planilhas oficiais do corpus.
# Track01 não é tratado automaticamente como o primeiro instrumento do nome.
FREE_SAX = [(0,229),(316,562),(573,730),(741,984),(993,1191),
            (1268,1481),(1495,1631),(1671,1820),(1827,1999),(2009,2158)]
FREE_CLARINET = [(0,215),(276,498),(571,788),(792,929),(944,1145),
                 (1149,1486),(1492,1721),(1748,1979),(1984,2059),(2063,2157)]
STANDARD_TRUMPET = [(2,169),(183,383),(385,617),(620,820),(823,950),
                    (960,1058),(1066,1187),(1198,1393),(1395,1549),(1550,1726)]
SOURCES = {
    "cello_melodico": dict(file="Excerpt 1.aif", origin="cellomondo", character="melódico"),
    "cello_tecnicas": dict(file="Excerpt 2.aif", origin="cellomondo", character="técnicas estendidas"),
    "cello_percussivo": dict(file="Excerpt 3.aif", origin="cellomondo", character="percussivo"),
    "somax_cassol": dict(file="Fabbrizio2c.wav", origin="Somax2", character="material creditado a Fabrizio Cassol"),
    "somax_joelle": dict(file="Joelle.wav", origin="Somax2", character="material creditado a Joëlle Léandre"),
    "somax_guitarra": dict(file="Marco_guitar.wav", origin="Somax2", character="material creditado a Marco Fiorini"),
    "duos_clarinet_alto": dict(file="F-Clarinet_AltoSax-Track01.wav", origin="Improvising Duos",
        character="improvisação livre; instrumento de Track01 por conferir", takes=FREE_CLARINET),
    "duos_sopsax_drums": dict(file="F-SopSax_Drums-Track01.wav", origin="Improvising Duos",
        character="improvisação livre; instrumento de Track01 por conferir", takes=FREE_SAX),
    "duos_trumpet_guitar": dict(file="S-Trumpet_ElecGuitar-Track01.wav", origin="Improvising Duos",
        character="Autumn Leaves; instrumento de Track01 por conferir", takes=STANDARD_TRUMPET),
}
# Exclusão explícita do usuário; catálogo completo preservado para rastreabilidade.
EXCLUDED_SOURCES = {"duos_trumpet_guitar": "Excluído pelo usuário em 18/10/2026."}
ACTIVE_SOURCES = {sid: spec for sid, spec in SOURCES.items() if sid not in EXCLUDED_SOURCES}


## 1. Funções e registro

Execute a célula abaixo uma vez. A análise guarda somente curvas reduzidas por blocos,
tempos e hashes em `dataset/revisao_v1.json`; não duplica os áudios longos.
As aprovações ficam nesse arquivo e podem ser retomadas em outra sessão.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def db(value):
    # O piso é só numérico, não um limiar de atividade.
    return float(20 * np.log10(max(float(value), 1e-15)))


def levels(audio):
    return dict(rms_dbfs=db(np.sqrt(np.mean(np.square(audio, dtype=np.float64)))),
                peak_dbfs=db(np.max(np.abs(audio))))


def file_info(path):
    info = sf.info(path)
    return dict(file=Path(path).name, frames=info.frames, sample_rate=info.samplerate,
                channels=info.channels, subtype=info.subtype, format=info.format,
                duration_s=info.duration, size_bytes=Path(path).stat().st_size)


def write_json(path, data, exclusive=False):
    # O rascunho pode ser atualizado; os resultados finais só são criados uma vez.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if exclusive:
        with path.open("x", encoding="utf-8") as stream:
            json.dump(data, stream, indent=2, ensure_ascii=False, allow_nan=False)
    else:
        temp = path.with_suffix(path.suffix + ".tmp")
        with temp.open("w", encoding="utf-8") as stream:
            json.dump(data, stream, indent=2, ensure_ascii=False, allow_nan=False)
        temp.replace(path)


def analyse_source(path, spec):
    info = file_info(path)
    power = []
    # 10 s de áudio por leitura; cada ponto de análise representa 100 ms.
    with sf.SoundFile(path) as stream:
        width = round(stream.samplerate * SELECTION["block_s"])
        while True:
            data = stream.read(width * 100, dtype="float64", always_2d=True)
            if not len(data):
                break
            if not np.isfinite(data).all():
                raise ValueError(f"Amostras não finitas: {path.name}")
            mono = data.mean(axis=1)
            complete = len(mono) // width
            if complete:
                power.extend(np.mean(mono[:complete * width].reshape(-1, width) ** 2, axis=1).tolist())
            # A fração final abaixo de 100 ms não entra no ranking de janelas completas.
    rms = np.sqrt(np.asarray(power))
    if not len(rms) or float(rms.max()) < 1e-12:
        raise ValueError(f"Fonte sem sinal mono utilizável: {path.name}")
    threshold = max(SELECTION["threshold_floor_dbfs"],
                    db(np.percentile(rms, 95)) - SELECTION["threshold_below_p95_db"])
    activity = 20 * np.log10(np.maximum(rms, 1e-15)) > threshold
    takes = spec.get("takes", [(0, info["duration_s"])])
    duration = SELECTION["duration_s"]
    step = SELECTION["step_s"]
    block_s = SELECTION["block_s"]
    candidates = []
    for take_index, (lo, hi) in enumerate(takes, 1):
        hi = min(hi, info["duration_s"])
        for tick in range(math.ceil(lo / step), math.floor((hi - duration) / step + 1e-8) + 1):
            start = tick * step
            first = round(start / block_s)
            last = first + round(duration / block_s)
            if last > len(rms):
                continue
            fraction = float(activity[first:last].mean())
            if fraction == 0:
                continue
            candidates.append(dict(start_s=start, end_s=start + duration, take=take_index,
                                   active_fraction=fraction,
                                   rms_dbfs=db(np.sqrt(np.mean(rms[first:last] ** 2)))))
    ranked = sorted(candidates, key=lambda c: (-c["active_fraction"], -c["rms_dbfs"], c["start_s"]))
    chosen = None
    # Procura também um primeiro candidato que permita um segundo não sobreposto.
    for first in ranked:
        second = next((c for c in ranked if c["end_s"] <= first["start_s"]
                       or c["start_s"] >= first["end_s"]), None)
        if second:
            chosen = sorted([first, second], key=lambda c: c["start_s"])
            break
    if chosen is None:
        raise ValueError(f"Não há dois intervalos ativos não sobrepostos em {path.name}.")
    info["sha256"] = sha256(path)
    return dict(source=info, spec=spec, threshold_dbfs=threshold, rms_curve=rms.tolist(),
                suggestions=chosen, starts_s=[c["start_s"] for c in chosen], approved=False)


def check_starts(item, starts):
    if len(starts) != 2 or not all(math.isfinite(float(t)) for t in starts):
        raise ValueError("Escolha exatamente dois inícios finitos em segundos.")
    starts = sorted(float(t) for t in starts)
    sr = item["source"]["sample_rate"]
    frames = item["source"]["frames"]
    length = round(SELECTION["duration_s"] * sr)
    bounds = item["spec"].get("takes", [(0, frames / sr)])
    snapped = []
    for start in starts:
        first = round(start * sr)
        if first < 0 or first + length > frames:
            raise ValueError("Trecho fora dos limites do arquivo.")
        actual = first / sr
        if not any(actual >= lo - 1 / sr and actual + length / sr <= hi + 1 / sr
                   for lo, hi in bounds):
            raise ValueError("O trecho atravessa uma pausa/limite de tomada documentada.")
        snapped.append(actual)
    if round(snapped[1] * sr) < round(snapped[0] * sr) + length:
        raise ValueError("Os dois trechos se sobrepõem.")
    return snapped


def prepare_clip(path, start_s):
    with sf.SoundFile(path) as stream:
        sr = stream.samplerate
        first = round(float(start_s) * sr)
        length = round(SELECTION["duration_s"] * sr)
        if first < 0 or first + length > stream.frames:
            raise ValueError("Trecho fora dos limites da fonte.")
        stream.seek(first)
        raw = stream.read(length, dtype="float64", always_2d=True)
    mono = raw.mean(axis=1)
    before = levels(mono)
    target_sr = PREPARATION["sample_rate"]
    divisor = math.gcd(sr, target_sr)
    # Filtro anti-alias da reamostragem; não há equalização artística.
    audio = resample_poly(mono, target_sr // divisor, sr // divisor,
                          window=tuple(PREPARATION["resample_window"]),
                          padtype=PREPARATION["resample_padtype"]) if sr != target_sr else mono.copy()
    expected = round(SELECTION["duration_s"] * target_sr)
    if len(audio) != expected:
        raise ValueError("Reamostragem não produziu a duração exata.")
    fade = round(PREPARATION["fade_ms"] * target_sr / 1000)
    envelope = np.linspace(0, 1, fade, endpoint=True)
    audio[:fade] *= envelope
    audio[-fade:] *= envelope[::-1]
    pre_gain = levels(audio)
    if not np.isfinite(audio).all() or np.max(np.abs(audio)) < 1e-12:
        raise ValueError("Trecho sem sinal utilizável após conversão mono.")
    wanted = PREPARATION["target_rms_dbfs"] - pre_gain["rms_dbfs"]
    allowed = PREPARATION["peak_ceiling_dbfs"] - pre_gain["peak_dbfs"]
    gain_db = min(wanted, allowed)
    audio *= 10 ** (gain_db / 20)
    return audio, dict(start_frame=first, end_frame_exclusive=first + length,
        start_s=first / sr, end_s=(first + length) / sr, input_mono_levels=before,
        levels_before_gain=pre_gain, gain_db=gain_db,
        peak_limited=bool(allowed < wanted - 1e-9), prepared_levels=levels(audio))


def run_analysis():
    if DRAFT.exists():
        data = json.loads(DRAFT.read_text())
        if data["selection"] != SELECTION or data["preparation"] != PREPARATION:
            raise ValueError("Configuração mudou: escolha outra VERSION para nova revisão.")
        if data.get("catalog") != json.loads(json.dumps(SOURCES)):
            raise ValueError("Catálogo mudou: escolha outra VERSION.")
    else:
        data = dict(version=VERSION, selection=SELECTION, preparation=PREPARATION,
                    catalog=SOURCES, sources={})
    data["excluded_sources"] = EXCLUDED_SOURCES
    data["active_source_ids"] = list(ACTIVE_SOURCES)
    for sid, spec in ACTIVE_SOURCES.items():
        path = ROOT / spec["file"]
        previous = data["sources"].get(sid)
        if not path.exists():
            print(sid, "— original ausente; revisão/exportações existentes preservadas.")
            continue
        if previous and previous["source"]["sha256"] == sha256(path):
            print(sid, "— revisão recuperada.")
            continue
        print(sid, "— analisando...", flush=True)
        data["sources"][sid] = analyse_source(path, spec)
        write_json(DRAFT, data)
    write_json(DRAFT, data)
    return data


def review_source(data, sid, starts=None, approve=False):
    if sid not in ACTIVE_SOURCES:
        raise ValueError("Fonte excluída ou fora da seleção ativa.")
    if sid not in data["sources"]:
        raise ValueError("Fonte ainda não analisada.")
    item = data["sources"][sid]
    path = ROOT / item["source"]["file"]
    if not path.exists():
        print("Original ausente. Use a seção 6 para escutar os cortes exportados.")
        return
    if sha256(path) != item["source"]["sha256"]:
        raise ValueError("A fonte mudou; refaça a análise antes da revisão.")
    times = check_starts(item, starts if starts is not None else item["starts_s"])
    # Prepara ambos antes de salvar a aprovação: uma conversão inválida não é aprovada.
    previews = [prepare_clip(path, start) for start in times]
    clear_output(wait=True)
    rms = np.asarray(item["rms_curve"])
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(np.arange(len(rms)) * SELECTION["block_s"],
            20 * np.log10(np.maximum(rms, 1e-15)), linewidth=0.6)
    ax.axhline(item["threshold_dbfs"], color="gray", linestyle="--", label="Limiar de atividade")
    for n, start in enumerate(times, 1):
        ax.axvspan(start, start + 10, alpha=0.25, label=f"Trecho {n}: {start:.3f}s")
    ax.set(title=sid, xlabel="Tempo na fonte (s)", ylabel="RMS mono / 100 ms (dBFS)")
    ax.legend()
    plt.show()
    plt.close(fig)
    for n, (audio, details) in enumerate(previews, 1):
        display(Markdown(f"**Trecho {n} — {times[n-1]:.3f} a {times[n-1]+10:.3f} s**"))
        print("Ganho (dB):", round(details["gain_db"], 3), "| RMS/pico:", details["prepared_levels"],
              "| Limitado pelo pico:", details["peak_limited"])
        display(Audio(audio, rate=48000, normalize=False))
    item["starts_s"] = times
    item["approved"] = bool(approve)
    item["reviewed_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(DRAFT, data)
    print("Aprovação registrada:", item["approved"], "|", sid)


def export_dataset(data, output=None):
    output = Path(output) if output is not None else OUTPUT
    if output.exists():
        raise FileExistsError(f"Não sobrescrevo {output}. Use outra VERSION.")
    if data.get("selection") != SELECTION or data.get("preparation") != PREPARATION:
        raise ValueError("A configuração da revisão não corresponde à configuração atual.")
    if not set(ACTIVE_SOURCES).issubset(data["sources"]):
        raise ValueError("Faltam fontes analisadas para a seleção ativa.")
    for sid in ACTIVE_SOURCES:
        item = data["sources"][sid]
        if not item["approved"]:
            raise ValueError(f"Ainda falta sua aprovação: {sid}")
        check_starts(item, item["starts_s"])
        path = ROOT / item["source"]["file"]
        if not path.exists() or sha256(path) != item["source"]["sha256"]:
            raise ValueError(f"Fonte ausente/alterada: {sid}")
    provenance = ROOT / "PROVENIENCIA.md"
    if not provenance.exists():
        raise FileNotFoundError("Falta PROVENIENCIA.md.")
    output.mkdir(parents=True, exist_ok=False)
    records = []
    jobs = [(sid, n, start, False) for sid in ACTIVE_SOURCES
            for n, start in enumerate(data["sources"][sid]["starts_s"], 1)]
    jobs.append(("cello_melodico", 0, 0.0, True))
    for sid, n, start, reference in jobs:
        source = data["sources"][sid]
        audio, details = prepare_clip(ROOT / source["source"]["file"], start)
        name = "Original.wav" if reference else f"{sid}__{n:02d}.wav"
        target = output / name
        # xb impede sobrescrita mesmo se surgir um arquivo durante a execução.
        with target.open("xb") as stream:
            sf.write(stream, audio, 48000, format="WAV", subtype="PCM_24")
        decoded, rate = sf.read(target, dtype="float64")
        if len(decoded) != 480000 or rate != 48000 or decoded.ndim != 1:
            raise ValueError(f"Falha de formato/duração: {name}")
        if np.max(np.abs(decoded - audio)) > 2 ** -22:
            raise ValueError(f"Falha na comparação PCM24: {name}")
        records.append(dict(file=name, sha256=sha256(target), source_id=sid,
            role="derived_reference" if reference else "test", include_in_calibration=True, details=details,
            file_info=file_info(target), measured_levels=levels(decoded)))
    notebook = ROOT / "curadoria_dataset.ipynb"
    manifest = dict(schema_version=1, version=data["version"],
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        selection=SELECTION, preparation=PREPARATION,
        sources={sid: data["sources"][sid] for sid in ACTIVE_SOURCES},
        excluded_sources=EXCLUDED_SOURCES, clips=records,
        source_reference=dict(file="Excerpt 1.aif", first_seconds=10,
                              independent_source=False, include_in_calibration=True),
        provenance_sha256=sha256(provenance),
        provenance_markdown=provenance.read_text(encoding="utf-8"),
        notebook_sha256=sha256(notebook),
        environment=dict(python=platform.python_version(), platform=platform.platform(),
            packages={p: importlib.metadata.version(p) for p in
                      ["numpy", "scipy", "soundfile", "matplotlib", "ipykernel"]}))
    report = validate_dataset(output, manifest, check_originals=True)
    manifest["validation"] = report
    write_json(output / "manifesto.json", manifest, exclusive=True)
    print("Exportação validada:", output)
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return manifest


def validate_dataset(folder, manifest=None, check_originals=False):
    folder = Path(folder)
    if manifest is None:
        manifest = json.loads((folder / "manifesto.json").read_text())
    tests = [c for c in manifest["clips"] if c["role"] == "test"]
    refs = [c for c in manifest["clips"] if c["role"] == "derived_reference"]
    expected_tests = 2 * len(manifest["sources"])
    if (len(tests) != expected_tests or len(refs) != 1
            or len({c["file"] for c in manifest["clips"]}) != expected_tests + 1
            or set(c["source_id"] for c in tests) != set(manifest["sources"])):
        raise ValueError("Esperados dois cortes por fonte ativa e uma referência distinta.")
    for sid in manifest["sources"]:
        pair = sorted([c for c in tests if c["source_id"] == sid],
                      key=lambda c: c["details"]["start_frame"])
        if len(pair) != 2 or pair[0]["details"]["end_frame_exclusive"] > pair[1]["details"]["start_frame"]:
            raise ValueError(f"Cortes ausentes ou sobrepostos: {sid}")
    size = 0
    for clip in manifest["clips"]:
        path = folder / clip["file"]
        info = file_info(path)
        audio, sr = sf.read(path, dtype="float64")
        if (info["frames"], sr, info["channels"], info["subtype"]) != (480000,48000,1,"PCM_24"):
            raise ValueError(f"Formato inválido: {path.name}")
        if sha256(path) != clip["sha256"] or not np.isfinite(audio).all():
            raise ValueError(f"Hash/sinal inválido: {path.name}")
        measured = levels(audio)
        if measured["peak_dbfs"] > PREPARATION["peak_ceiling_dbfs"] + 0.00002:
            raise ValueError(f"Pico acima do teto: {path.name}")
        if not clip["details"]["peak_limited"] and abs(measured["rms_dbfs"] + 24) > 0.001:
            raise ValueError(f"RMS fora do alvo: {path.name}")
        for key in ["rms_dbfs", "peak_dbfs"]:
            if abs(measured[key] - clip["measured_levels"][key]) > 1e-8:
                raise ValueError(f"Medida inconsistente: {path.name}")
        size += path.stat().st_size
    reference_checked = False
    if check_originals:
        ref = refs[0]
        original = ROOT / manifest["sources"][ref["source_id"]]["source"]["file"]
        prepared, _ = prepare_clip(original, 0)
        decoded, _ = sf.read(folder / ref["file"], dtype="float64")
        if not np.allclose(decoded, prepared, rtol=0, atol=2 ** -22):
            raise ValueError("Referência não corresponde ao início de Excerpt 1.")
        reference_checked = True
    # A cópia textual dentro do manifesto preserva os créditos mesmo após mover as fontes.
    text_hash = hashlib.sha256(manifest["provenance_markdown"].encode("utf-8")).hexdigest()
    if text_hash != manifest["provenance_sha256"]:
        raise ValueError("Proveniência inconsistente.")
    return dict(status="passed", test_clips=len(tests), derived_references=len(refs),
                wav_bytes=size, wav_mb=round(size / 1e6, 2),
                peak_limited_clips=sum(c["details"]["peak_limited"] for c in manifest["clips"]),
                reference_compared_with_source=reference_checked,
                validation_requires_originals=bool(check_originals))

## 2. Inventário

Somente os oito arquivos do catálogo alimentam a seleção. `Original.wav` atual aparece
como referência curta preservada. Não serão descobertos recursivamente os WAVs já exportados.

In [ ]:
for sid, spec in ACTIVE_SOURCES.items():
    path = ROOT / spec["file"]
    if path.exists():
        info = file_info(path)
        print(f"{sid:24} {info['duration_s']:8.2f} s | {info['channels']} ch | "
              f"{info['sample_rate']} Hz | {info['subtype']}")
    else:
        print(sid, "— original ausente")
if (ROOT / "Original.wav").exists():
    print("Referência curta atual, preservada:", file_info(ROOT / "Original.wav"))

## 3. Sugerir ou recuperar os cortes

Na primeira execução, percorre as fontes em blocos, calcula os candidatos e grava a
revisão. Em execuções seguintes, recupera tempos e aprovações se os hashes coincidirem.
Uma fonte alterada perde a aprovação anterior e é analisada novamente.

In [ ]:
review = run_analysis()
for sid in ACTIVE_SOURCES:
    item = review["sources"][sid]
    print(sid, "| inícios:", item["starts_s"], "| aprovado:", item["approved"])

## 4. Escutar, ajustar e aprovar — uma fonte de cada vez

1. Copie um identificador da lista acima para `SOURCE_ID`.
2. Execute com `APPROVE = False` e escute os dois players. São prévias com o mesmo preparo
   da exportação; mudar o volume do computador não muda os arquivos.
3. Para ajustar, coloque `STARTS = [inicio_1, inicio_2]`, em segundos, e execute novamente.
   `None` conserva os tempos atualmente registrados. Cortes sobrepostos ou fora de uma
   tomada são recusados.
4. Depois de ouvir, mude para `APPROVE = True` e execute para guardar sua aprovação.
5. Para a próxima fonte, mude o identificador e volte a `APPROVE = False` **antes** de escutar.

O gráfico é da fonte antes da normalização, já considerando a conversão para mono.
Cada execução substitui os dois players anteriores na célula; não reúne todas as prévias.
Para a referência extra, o início de Excerpt 1 é fixo: escute também o player próprio abaixo.

In [ ]:
SOURCE_ID = "duos_sopsax_drums"
STARTS = None  # Exemplo: [2.0, 25.0]; use apenas tempos que você realmente revisou.
APPROVE = True  # False para apenas registrar os inícios sem aprovar.
review_source(review, SOURCE_ID, starts=STARTS, approve=APPROVE)

In [ ]:
reference_source = ROOT / "Excerpt 1.aif"
if reference_source.exists():
    reference_audio, reference_details = prepare_clip(reference_source, 0)
    display(Markdown("**Referência extra: primeiros 10 s de Excerpt 1 (não é outra fonte).**"))
    display(Audio(reference_audio, rate=48000, normalize=False))
else:
    print("Fonte original ausente; a referência exportada pode ser ouvida na seção 6.")

## 5. Exportar após as nove aprovações

Confira a lista abaixo. Só então altere `EXPORT_APPROVED` para `True`.
A referência adicional acompanha a exportação. O destino `dataset/v1/` não pode existir;
para outra versão, altere `VERSION` na configuração e execute novamente as seções.
Se a exportação for interrompida, arquivos parciais permanecem identificáveis e não serão
sobrescritos. Use outra versão após conferir o ocorrido.

Esta célula verifica todos os hashes das fontes antes de começar e reabre cada WAV para
conferir o PCM exportado. O manifesto só é criado após a validação completa.

In [ ]:
for sid in ACTIVE_SOURCES:
    item = review["sources"].get(sid, {})
    print(sid, "—", "APROVADO" if item.get("approved") else "PENDENTE")

EXPORT_APPROVED = False
if EXPORT_APPROVED:
    manifest = export_dataset(review)
else:
    print("Nenhum WAV definitivo exportado. Revise/aprove as oito fontes antes de exportar.")

## 6. Conferir e escutar o dataset, inclusive sem os originais

Execute a configuração e as funções e depois esta seção. Ela não depende das fontes
longas: usa os WAVs exportados, o manifesto e a proveniência nele preservada.
Validação técnica não substitui sua escuta nem a conferência das condições de uso.

Depois de conferir os 19 arquivos e manter uma cópia segura do dataset, você poderá
retirar manualmente os originais da máquina. Sem recuperá-los, não será possível refazer
novos cortes. O notebook nunca apaga ou move fontes.

In [ ]:
if (OUTPUT / "manifesto.json").exists():
    saved_manifest = json.loads((OUTPUT / "manifesto.json").read_text())
    print(json.dumps(validate_dataset(OUTPUT), ensure_ascii=False, indent=2))
    for clip in saved_manifest["clips"]:
        print(clip["file"], "|", clip["role"], "| RMS:", round(clip["measured_levels"]["rms_dbfs"], 3),
              "| pico limitado:", clip["details"]["peak_limited"])
else:
    print("Dataset ainda não exportado para", OUTPUT)

In [ ]:
# Escolha só um arquivo por vez; a lista está na célula anterior.
LISTEN_FILE = "Original.wav"
if (OUTPUT / "manifesto.json").exists():
    allowed_files = {clip["file"] for clip in saved_manifest["clips"]}
    if LISTEN_FILE not in allowed_files:
        raise ValueError("Escolha um nome listado no manifesto.")
    audio, sr = sf.read(OUTPUT / LISTEN_FILE)
    display(Audio(audio, rate=sr, normalize=False))
else:
    print("Exporte primeiro os cortes aprovados.")